In [ ]:
import os

This notebook walks through the failure mode where `reader/app.js` POSTs to `/api/generate` and receives a `404 Not Found` response from the deployed Vercel site.

Outline:
1. Verify failed POST request and 404 response
2. Inspect frontend API URL construction
3. Check server route and deployment path
4. Add fallback endpoint configuration
5. Implement retry and error handling in JavaScript
6. Test TTS API with a local mock server
</VSCode.Cell>
<VSCode.Cell language="markdown">## 1. Verify failed POST request and 404 response

Inspect browser console messages and network logs to confirm the POST request and exact URL. This verifies whether the failure is a missing route or a bad endpoint path.

The example error from the deployed app is:

- `POST https://sesbot-okuyucu-5blxpuuq6-hakancineli1.vercel.app/api/generate 404 (Not Found)`
- `Sunucu hatasi: 404 The page could not be found`
</VSCode.Cell>
<VSCode.Cell language="markdown">## 2. Inspect frontend API URL construction

Read `reader/app.js` to locate the `generateAudio` call and ensure the endpoint is built correctly.

Key checks:
- Endpoint path should be `/api/generate`
- No deployment-only absolute URL should be hardcoded improperly
- The frontend should support both local and deployed environment paths
</VSCode.Cell>
<VSCode.Cell language="markdown">## 3. Check server route and deployment path

Verify the backend defines `/api/generate` and confirm the Vercel deployment package includes `api/generate.py` and `requirements.txt`.

If the route does not exist in the deployed package, the frontend will always receive 404.
</VSCode.Cell>
<VSCode.Cell language="markdown">## 4. Add fallback endpoint configuration

Implement a configuration option for the frontend API base URL so the app can switch between local and deployed endpoints without hardcoding.

Recommended approach:
- Detect `window.location.hostname`
- Use `/api/generate` for same-origin
- Use a configured fallback for deployed environments if needed
</VSCode.Cell>
<VSCode.Cell language="markdown">## 5. Implement retry and error handling in JavaScript

Update `generateAudio` and `generatePageAudio` to:
- Catch 404 errors cleanly
- Show a helpful message
- Optionally retry with a corrected base URL if a deploy domain mismatch is detected
</VSCode.Cell>
<VSCode.Cell language="markdown">## 6. Test TTS API with a local mock server

Create a minimal local mock server that responds to `/api/generate` and verify the frontend can call it successfully.

This confirms the client-side logic is correct independent of Vercel deployment issues.


In [ ]:
from pathlib import Path

repo = Path('/Users/hakan/Desktop/sesbot')
script = repo / 'scripts' / 'deploy_vercel.py'

print('Running deploy script...')
result = subprocess.run([
    'python3', str(script),
    '--token', os.environ.get('VERCEL_TOKEN'),
    '--skip-build'
], cwd=repo, capture_output=True, text=True)
print('returncode=', result.returncode)
print('stdout:\n', result.stdout)
print('stderr:\n', result.stderr)
